# Lab 03 — Linear Models

**Guide:** `practice/03_linear_models.pdf` explains every task. Read it alongside.

Cells marked **TODO** are yours: replace `raise NotImplementedError` with your code. Every other cell is ready and
draws the plots once your functions work. Run the notebook from top to bottom.

**The rule of this lab:** every model is a loss written with `jax.numpy`; the gradient always comes from
`jax.grad` / `jax.value_and_grad` (through `train`), never from a hand-derived formula. Each model is then checked
against its scikit-learn twin: same split, same scaling, and a printed maximum difference.

| Part | Task | sklearn twin | Time |
|:--|:--|:--|:-:|
| A | Data, split methods, scaling, metrics from scratch | `metrics`, `train_test_split` | 20 min |
| B | Linear regression: MSE loss, `jax.grad`, normal equation | `LinearRegression` | 15 min |
| C | Polynomial features, ridge, lasso and a proximal step | `PolynomialFeatures`, `Ridge`, `Lasso` | 30 min |
| D | The perceptron: its update is a `jax.grad` | `Perceptron` | 15 min |
| E | Logistic regression on spam, thresholds and metrics | `LogisticRegression` | 20 min |
| F | Challenge: one protocol, three models | `cross_val_score` | home |

In [ ]:
import itertools

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from lab03_utils import (
    compare,
    kfold_indices,
    load_housing,
    load_spam,
    make_blobs,
    make_xor,
    plot_boundary,
    plot_confusion,
    plot_degree_curve,
    plot_history,
    plot_path,
    plot_thresholds,
    plot_validation_curve,
    train,
)
from sklearn import metrics as skm
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Perceptron, Ridge
from sklearn.preprocessing import PolynomialFeatures

plt.rcParams["figure.figsize"] = (9, 4)
rng = np.random.default_rng(0)

## The two datasets

* **House prices** (regression): California housing, 20640 districts, 8 features, target = median house value in
  100 000 USD. Four skewed features are already replaced by their logarithm in `load_housing()`.
* **Spam** (classification): Spambase, 4601 e-mails, 57 features (word and character frequencies, capital-run lengths),
  39% spam. `load_spam()` returns the raw frequencies; `y = 1` is spam.

In [ ]:
X_h, y_h, names_h = load_housing()
X_s, y_s, names_s = load_spam()
print("housing:", X_h.shape, "target mean", y_h.mean().round(3), "| spam:", X_s.shape, "spam fraction", y_s.mean().round(3))

## Part A — Data, splits and metrics

### A1. A stratified split

Implement `stratified_split(y, test_frac, rng)`: return `(train_idx, test_idx)` such that **each class** keeps the
same proportion in both sets (guide, task A1).

In [ ]:
def stratified_split(y, test_frac, rng) -> tuple[np.ndarray, np.ndarray]:
    """TODO: return (train_idx, test_idx); every class keeps the same proportion in both."""
    raise NotImplementedError

In [ ]:
# Housing: an ordinary random 80/20 split. Spam: stratified 80/20.
perm = rng.permutation(len(y_h))
te_h, tr_h = perm[: len(perm) // 5], perm[len(perm) // 5 :]
tr_s, te_s = stratified_split(y_s, 0.2, rng)
print("housing train/test:", len(tr_h), len(te_h))
print("spam train/test:", len(tr_s), len(te_s), "| spam fraction train", y_s[tr_s].mean().round(4), "test", y_s[te_s].mean().round(4))

# Why stratify? On 100 e-mails, how different are the spam fractions of the two parts?
def gap(stratify, n_rep=300):
    gaps = []
    for _ in range(n_rep):
        sub = y_s[rng.choice(len(y_s), 100, replace=False)]
        tr, te = stratified_split(sub, 0.2, rng) if stratify else (lambda p: (p[20:], p[:20]))(rng.permutation(100))
        gaps.append(abs(sub[tr].mean() - sub[te].mean()))
    return np.mean(gaps)

print(f"mean |train - test| spam fraction: random {gap(False):.3f}, stratified {gap(True):.3f}")

### A2. Scaling fitted on the training set only

Implement `fit_standardizer(X)` (per-column mean and standard deviation; a constant column gets standard deviation 1)
and `standardize(X, mean, std)`. The mean and the deviation are **learned from the training set** and applied to both
sets. Fitting them on all the data leaks information from the test set (Class 01, data leakage).

In [ ]:
def fit_standardizer(X) -> tuple[np.ndarray, np.ndarray]:
    """TODO: return (mean, std) per column; a constant column gets std 1."""
    raise NotImplementedError


def standardize(X, mean, std) -> np.ndarray:
    """TODO: return (X - mean) / std."""
    raise NotImplementedError

In [ ]:
mean_h, std_h = fit_standardizer(X_h[tr_h])
Xtr_h, Xte_h = standardize(X_h[tr_h], mean_h, std_h), standardize(X_h[te_h], mean_h, std_h)
ytr_h, yte_h = y_h[tr_h], y_h[te_h]

L_s = np.log1p(X_s)
mean_s, std_s = fit_standardizer(L_s[tr_s])
Xtr_s, Xte_s = standardize(L_s[tr_s], mean_s, std_s), standardize(L_s[te_s], mean_s, std_s)
ytr_s, yte_s = y_s[tr_s], y_s[te_s]
print("train means ~0:", Xtr_h.mean(axis=0).round(2)[:4], "| test means differ a little:", Xte_h.mean(axis=0).round(2)[:4])

### A3. Regression metrics

Implement `regression_metrics(y, y_hat)` returning a dictionary with `mse`, `rmse`, `mae`, `smape` (in %) and `r2`:

$$\text{sMAPE} = \frac{100}{n}\sum_i \frac{2\,|y_i - \hat y_i|}{|y_i| + |\hat y_i|} \qquad
R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$$

In [ ]:
def regression_metrics(y, y_hat) -> dict[str, float]:
    """TODO: return a dict with mse, rmse, mae, smape (in %) and r2."""
    raise NotImplementedError

In [ ]:
# Check against scikit-learn (sMAPE has no sklearn function: verify it by hand on two points).
y_t, y_p = np.array([100.0, 200.0, 50.0, 80.0]), np.array([110.0, 180.0, 55.0, 60.0])
ours = regression_metrics(y_t, y_p)
compare("mse", ours["mse"], skm.mean_squared_error(y_t, y_p))
compare("mae", ours["mae"], skm.mean_absolute_error(y_t, y_p))
compare("r2", ours["r2"], skm.r2_score(y_t, y_p))
print("smape:", round(ours["smape"], 4), "by hand:", round(100 * np.mean([20 / 210, 40 / 380, 10 / 105, 40 / 140]), 4))
print("baseline (predict the training mean) on the housing test set:")
print({k: round(v, 3) for k, v in regression_metrics(yte_h, np.full(len(yte_h), ytr_h.mean())).items()})

### A4. Classification metrics

Implement `confusion_counts(y, y_hat)` returning `(tn, fp, fn, tp)` and `classification_metrics(y, y_hat)` returning
`accuracy`, `precision`, `recall`, `f1` and `mcc`. Return 0 when a denominator is 0.

$$\text{MCC} = \frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

In [ ]:
def confusion_counts(y, y_hat) -> tuple[int, int, int, int]:
    """TODO: return (tn, fp, fn, tp) for labels 0/1."""
    raise NotImplementedError


def classification_metrics(y, y_hat) -> dict[str, float]:
    """TODO: return a dict with accuracy, precision, recall, f1 and mcc (0 when a denominator is 0)."""
    raise NotImplementedError

In [ ]:
y_t = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_p = np.array([1, 1, 1, 0, 1, 0, 0, 0, 0, 0])
ours = classification_metrics(y_t, y_p)
compare("confusion matrix", confusion_counts(y_t, y_p), skm.confusion_matrix(y_t, y_p).ravel())
compare("accuracy", ours["accuracy"], skm.accuracy_score(y_t, y_p))
compare("precision", ours["precision"], skm.precision_score(y_t, y_p))
compare("recall", ours["recall"], skm.recall_score(y_t, y_p))
compare("f1", ours["f1"], skm.f1_score(y_t, y_p))
compare("mcc", ours["mcc"], skm.matthews_corrcoef(y_t, y_p))

# A useless model: call every e-mail spam.
all_spam = classification_metrics(yte_s, np.ones_like(yte_s))
print({k: round(v, 3) for k, v in all_spam.items()})

## Part B — Linear regression

$\hat y = w^\top x + b$, trained by minimizing the mean squared error. Parameters are a dictionary
`{"w": (d,), "b": ()}`, the format `train` expects.

### B1. Model, loss and normal equation

Implement `init_params(d)` (zeros), `predict_linear(params, X)`, `mse_loss(params, X, y)` (written with `jax.numpy`;
**no gradient formulas**) and `normal_equation(X, y)`: append a column of ones to $X$ and solve
$(A^\top A)\,\theta = A^\top y$ with `np.linalg.solve`. It returns `(w, b)`.

In [ ]:
def init_params(d) -> dict[str, jax.Array]:
    """TODO: {"w": zeros(d), "b": 0} as JAX arrays."""
    raise NotImplementedError


def predict_linear(params, X) -> jax.Array:
    """TODO: X @ w + b."""
    raise NotImplementedError


def mse_loss(params, X, y) -> jax.Array:
    """TODO: mean squared error, written with jax.numpy."""
    raise NotImplementedError


def normal_equation(X, y) -> tuple[np.ndarray, float]:
    """TODO: solve (A^T A) theta = A^T y with A = [X, 1]; return (w, b)."""
    raise NotImplementedError

In [ ]:
params, hist = train(mse_loss, init_params(8), Xtr_h, ytr_h, lr=0.05, steps=1000)
sk = LinearRegression().fit(Xtr_h, ytr_h)
w_ne, b_ne = normal_equation(Xtr_h, ytr_h)

compare("jax.grad vs sklearn (weights)", params["w"], sk.coef_)
compare("jax.grad vs sklearn (bias)", params["b"], sk.intercept_)
compare("normal equation vs sklearn", np.r_[w_ne, b_ne], np.r_[sk.coef_, sk.intercept_])
for name, pred in [
    ("mean baseline", np.full(len(yte_h), ytr_h.mean())),
    ("ours (jax.grad)", np.asarray(predict_linear(params, Xte_h))),
    ("sklearn", sk.predict(Xte_h)),
]:
    print(f"{name:<16}", {k: round(v, 3) for k, v in regression_metrics(yte_h, pred).items()})

In [ ]:
# The same model on the raw (unstandardized) features, with the same optimizer and settings.
_, hist_raw = train(mse_loss, init_params(8), X_h[tr_h], ytr_h, lr=0.05, steps=1000)
plot_history({"standardized": hist, "raw features": hist_raw})
print("final training loss: standardized", hist[-1].round(4), "| raw", hist_raw[-1].round(4))

## Part C — Polynomial features and regularization

### C1. Polynomial feature expansion

Implement `polynomial_features(X, degree)`: all monomials of the columns with degrees $1,\dots,\text{degree}$
(no constant column), ordered by degree and, inside a degree, by `itertools.combinations_with_replacement`. Check it
against `PolynomialFeatures(degree, include_bias=False)`.

In [ ]:
def polynomial_features(X, degree) -> np.ndarray:
    """TODO: all monomials of degree 1..degree, columns ordered like sklearn's PolynomialFeatures."""
    raise NotImplementedError

In [ ]:
for d in (2, 3):
    compare(f"degree {d} vs PolynomialFeatures", polynomial_features(X_h[:50], d), PolynomialFeatures(d, include_bias=False).fit_transform(X_h[:50]))
print("number of features for degrees 1-4:", [polynomial_features(X_h[:5], d).shape[1] for d in range(1, 5)])

Now the problem: with only **300 training examples**, fit the linear model on the expanded features for degrees 1 to 4
(scikit-learn's `LinearRegression` solves it exactly; the expanded columns are standardized on the training part).

In [ ]:
sub = rng.choice(tr_h, 300, replace=False)
train_mse, test_mse = [], []
for d in range(1, 5):
    P, Pt = polynomial_features(X_h[sub], d), polynomial_features(X_h[te_h], d)
    m_, s_ = fit_standardizer(P)
    model = LinearRegression().fit(standardize(P, m_, s_), y_h[sub])
    train_mse.append(regression_metrics(y_h[sub], model.predict(standardize(P, m_, s_)))["mse"])
    test_mse.append(regression_metrics(y_h[te_h], model.predict(standardize(Pt, m_, s_)))["mse"])
    print(f"degree {d}: {P.shape[1]:4d} features | train MSE {train_mse[-1]:.4f} | test MSE {test_mse[-1]:.4g}")
plot_degree_curve(range(1, 5), train_mse, test_mse)

### C2. Ridge (L2) and lasso (L1) losses

Implement `ridge_loss(params, X, y, lam)` $= \text{MSE} + \lambda\lVert w\rVert_2^2$ and
`lasso_loss(params, X, y, lam)` $= \text{MSE} + \lambda\lVert w\rVert_1$. The bias is **not** penalized. Pass
`lam` to `train` as data, so JAX compiles the loss once for all values of $\lambda$. Then choose $\lambda$ by 5-fold
cross-validation on the 300 examples (degree 2, 44 features) and compare with scikit-learn.

*Conventions:* scikit-learn's `Ridge(alpha)` minimizes $\lVert y - Xw\rVert^2 + \alpha\lVert w\rVert^2$ (a *sum*), so
$\alpha = \lambda N$. Its `Lasso(alpha)` minimizes $\frac1{2N}\lVert y - Xw\rVert^2 + \alpha\lVert w\rVert_1$, so
$\alpha = \lambda/2$.

In [ ]:
def ridge_loss(params, X, y, lam) -> jax.Array:
    """TODO: MSE + lam * ||w||_2^2 (the bias is not penalized)."""
    raise NotImplementedError


def lasso_loss(params, X, y, lam) -> jax.Array:
    """TODO: MSE + lam * ||w||_1 (the bias is not penalized)."""
    raise NotImplementedError

In [ ]:
P2, P2_test = polynomial_features(X_h[sub], 2), polynomial_features(X_h[te_h], 2)
lams = np.logspace(-4, 0.5, 10)
folds = kfold_indices(len(sub), 5, rng)

train_curve, val_curve = [], []
for lam in lams:
    tr_e, va_e = [], []
    for tr_i, va_i in folds:
        m_, s_ = fit_standardizer(P2[tr_i])
        p, _ = train(ridge_loss, init_params(P2.shape[1]), standardize(P2[tr_i], m_, s_), y_h[sub][tr_i], lam, lr=0.05, steps=1500)
        tr_e.append(regression_metrics(y_h[sub][tr_i], np.asarray(predict_linear(p, standardize(P2[tr_i], m_, s_))))["mse"])
        va_e.append(regression_metrics(y_h[sub][va_i], np.asarray(predict_linear(p, standardize(P2[va_i], m_, s_))))["mse"])
    train_curve.append(np.mean(tr_e))
    val_curve.append(np.mean(va_e))
best = lams[int(np.argmin(val_curve))]
plot_validation_curve(lams, train_curve, val_curve)
print(f"best lambda by 5-fold CV: {best:.4g}")

In [ ]:
# Refit on the 300 examples with the best lambda and compare with scikit-learn's Ridge (alpha = lambda * N).
m2, s2 = fit_standardizer(P2)
A2, A2_test = standardize(P2, m2, s2), standardize(P2_test, m2, s2)
ridge_p, _ = train(ridge_loss, init_params(44), A2, y_h[sub], best, lr=0.05, steps=3000)
ridge_sk = Ridge(alpha=best * len(sub)).fit(A2, y_h[sub])
compare("ridge weights", ridge_p["w"], ridge_sk.coef_)
lin_sk = LinearRegression().fit(A2, y_h[sub])
print("test MSE on 300 examples, degree 2: plain", round(regression_metrics(y_h[te_h], lin_sk.predict(A2_test))["mse"], 4),
      "| ridge", round(regression_metrics(y_h[te_h], np.asarray(predict_linear(ridge_p, A2_test)))["mse"], 4),
      "| best CV MSE", round(min(val_curve), 4))

# Regularization paths: coefficients versus lambda.
path = np.array([np.asarray(train(ridge_loss, init_params(44), A2, y_h[sub], lam, lr=0.05, steps=1500)[0]["w"]) for lam in lams])
plot_path(lams, path, "ridge: coefficients shrink smoothly towards 0")

In [ ]:
# The same 300 examples with degree-3 features (164 columns): the plain fit exploded in C1, ridge tames it.
P3, P3_test = polynomial_features(X_h[sub], 3), polynomial_features(X_h[te_h], 3)
m3, s3 = fit_standardizer(P3)
B3, B3_test = standardize(P3, m3, s3), standardize(P3_test, m3, s3)
plain3 = LinearRegression().fit(B3, y_h[sub])
ridge3, _ = train(ridge_loss, init_params(164), B3, y_h[sub], 1.0, lr=0.05, steps=3000)
print("degree 3, 300 examples, test MSE: plain", round(regression_metrics(y_h[te_h], plain3.predict(B3_test))["mse"], 3),
      "| ridge (lambda = 1)", round(regression_metrics(y_h[te_h], np.asarray(predict_linear(ridge3, B3_test)))["mse"], 3))

In [ ]:
# Lasso by jax.grad (a subgradient): compare with scikit-learn's coordinate descent.
lam = 0.03
lasso_p, _ = train(lasso_loss, init_params(44), A2, y_h[sub], lam, lr=0.02, steps=6000)
lasso_sk = Lasso(alpha=lam / 2, max_iter=1_000_000, tol=1e-12).fit(A2, y_h[sub])


def objective(w, b):
    return float(np.mean((A2 @ w + b - y_h[sub]) ** 2) + lam * np.abs(w).sum())


compare("lasso weights", lasso_p["w"], lasso_sk.coef_)
compare("lasso objective value", objective(np.asarray(lasso_p["w"]), float(lasso_p["b"])), objective(lasso_sk.coef_, lasso_sk.intercept_))
compare("lasso test predictions", predict_linear(lasso_p, A2_test), lasso_sk.predict(A2_test))
w = np.asarray(lasso_p["w"])
print(f"exactly zero: ours {int(np.sum(w == 0))} | sklearn {int(np.sum(lasso_sk.coef_ == 0))} of 44")
print(f"below 1e-3:   ours {int(np.sum(np.abs(w) < 1e-3))} | sklearn {int(np.sum(np.abs(lasso_sk.coef_) < 1e-3))}")

### C3. A proximal step gives exact zeros

Implement `soft_threshold(w, t)` $= \text{sign}(w)\max(|w| - t, 0)$ and `proximal_lasso(X, y, lam, lr, steps)`. Each
iteration takes a gradient step on the **smooth part** (`jax.grad(mse_loss)`), then soft-thresholds the weights by
$\text{lr}\cdot\lambda$ (the bias is not thresholded). This is ISTA, *proximal* gradient descent.

In [ ]:
def soft_threshold(w, t) -> jax.Array:
    """TODO: sign(w) * max(|w| - t, 0)."""
    raise NotImplementedError


def proximal_lasso(X, y, lam, lr, steps) -> dict[str, jax.Array]:
    """TODO: ISTA. Gradient step on the MSE with jax.grad, then soft-threshold the weights by lr * lam."""
    raise NotImplementedError

In [ ]:
# The MSE gradient is L-Lipschitz with L = 2 * lambda_max(A^T A / N) (its Hessian): the step 1/L is the safe choice.
lipschitz = 2 * np.linalg.eigvalsh(A2.T @ A2 / len(A2)).max()
prox_p = proximal_lasso(A2, y_h[sub], lam, 1 / lipschitz, 100_000)
print(f"L = {lipschitz:.1f}, step 1/L = {1 / lipschitz:.4f}")
compare("proximal lasso weights", prox_p["w"], lasso_sk.coef_)
compare("proximal lasso objective value", objective(np.asarray(prox_p["w"]), float(prox_p["b"])), objective(lasso_sk.coef_, lasso_sk.intercept_))
compare("proximal lasso test predictions", predict_linear(prox_p, A2_test), lasso_sk.predict(A2_test))
print(f"exactly zero: proximal {int(np.sum(np.asarray(prox_p['w']) == 0))} | sklearn {int(np.sum(lasso_sk.coef_ == 0))} of 44")
kept = [i for i in range(44) if np.asarray(prox_p["w"])[i] != 0]
print("test MSE: lasso", round(regression_metrics(y_h[te_h], np.asarray(predict_linear(prox_p, A2_test)))["mse"], 4), "with", len(kept), "features")

## Part D — The perceptron

$\hat y = \text{sign}(w^\top x + b)$ with labels $y\in\{-1,+1\}$. Its loss is the **perceptron criterion**
$\ell = \max(0,\,-m)$ with the margin $m = y\,(w^\top x + b)$: zero for a correct point, the distance-like amount
$-m$ for a mistake.

### D1. The loss and the online rule

Implement `perceptron_loss(params, X, y)` (mean over the rows; use `jnp.where(margin <= 0, -margin, 0.0)`, so a point
**on** the boundary counts as a mistake) and `train_perceptron(X, y, epochs)`. It visits the examples **in order**,
computes `jax.grad(perceptron_loss)` of **one example**, and updates $\theta \leftarrow \theta - g$ (learning
rate 1). It returns the parameters and the number of mistakes in every epoch (a mistake is a non-zero gradient).

In [ ]:
def perceptron_loss(params, X, y) -> jax.Array:
    """TODO: mean of max(0, -margin) with margin = y * (X @ w + b); a margin of exactly 0 is a mistake."""
    raise NotImplementedError


def train_perceptron(X, y, epochs) -> tuple[dict[str, jax.Array], list[int]]:
    """TODO: online rule. For every example in order: g = jax.grad(perceptron_loss) of that one example, params -= g.

    Return (params, mistakes_per_epoch)."""
    raise NotImplementedError

In [ ]:
def predict_class(params, X):
    return (np.asarray(X @ params["w"] + params["b"]) > 0).astype(int)


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
Xb, yb = make_blobs(rng, 100, gap=4.0)
p_sep, mist_sep = train_perceptron(Xb, 2 * yb - 1, 15)
plot_boundary(lambda Z: predict_class(p_sep, Z), Xb, yb, f"separable: mistakes/epoch {mist_sep[:4]}...", ax=axes[0])
Xo, yo = make_blobs(rng, 100, gap=1.5)
p_ov, mist_ov = train_perceptron(Xo, 2 * yo - 1, 15)
plot_boundary(lambda Z: predict_class(p_ov, Z), Xo, yo, f"overlapping: {mist_ov[-3:]} per epoch", ax=axes[1])
Xx, yx = make_xor(rng, 200)
p_xor, mist_xor = train_perceptron(Xx, 2 * yx - 1, 20)
plot_boundary(lambda Z: predict_class(p_xor, Z), Xx, yx, f"XOR: accuracy {np.mean(predict_class(p_xor, Xx) == yx):.2f}", ax=axes[2])
plt.show()
print("separable: mistakes per epoch", mist_sep)
print("overlapping: mistakes per epoch", mist_ov)
print("XOR (raw features): accuracy", np.mean(predict_class(p_xor, Xx) == yx), "| mistakes per epoch", mist_xor[:6], "...")

# XOR becomes separable with the degree-2 features (x1, x2, x1^2, x1 x2, x2^2).
P_xor = polynomial_features(Xx, 2)
p_xp, mist_xp = train_perceptron(P_xor, 2 * yx - 1, 20)
print("XOR with polynomial features: mistakes per epoch", mist_xp, "| accuracy", np.mean(predict_class(p_xp, P_xor) == yx))

In [ ]:
# On spam (standardized log features), one pass over the training set, against scikit-learn's Perceptron.
p_spam, mist_spam = train_perceptron(Xtr_s, 2 * ytr_s - 1, 1)
sk_p = Perceptron(penalty=None, eta0=1.0, shuffle=False, tol=None, max_iter=1).fit(Xtr_s, ytr_s)  # pyright: ignore[reportArgumentType]
compare("perceptron weights", p_spam["w"], np.asarray(sk_p.coef_)[0])
compare("perceptron bias", p_spam["b"], np.asarray(sk_p.intercept_)[0])
print("mistakes in the epoch:", mist_spam[0], "of", len(ytr_s))
print({k: round(v, 3) for k, v in classification_metrics(yte_s, predict_class(p_spam, Xte_s)).items()})

## Part E — Logistic regression

$p(y=1\mid x) = \sigma(w^\top x + b)$ with $\sigma(z) = 1/(1+e^{-z})$. Training minimizes the **binary
cross-entropy** (the negative log-likelihood of the labels).

### E1. Loss, probabilities and regularization

Implement `sigmoid(z)`, `bce_loss(params, X, y)` for labels $y\in\{0,1\}$ and `bce_l2_loss(params, X, y, lam)`
$= \text{BCE} + \frac\lambda2\lVert w\rVert^2$. Write the loss with the *logits* $z$ and `jnp.logaddexp`:

$$\ell = \log(1 + e^{z}) - y\,z$$

Train it on the standardized spam features. scikit-learn's `LogisticRegression(C)` minimizes
$C\sum_i\ell_i + \frac12\lVert w\rVert^2$, so our $\lambda$ corresponds to $C = 1/(\lambda N)$.

In [ ]:
def sigmoid(z) -> jax.Array:
    """TODO: 1 / (1 + exp(-z))."""
    raise NotImplementedError


def bce_loss(params, X, y) -> jax.Array:
    """TODO: mean of logaddexp(0, z) - y * z, with z = X @ w + b and y in {0, 1}."""
    raise NotImplementedError


def bce_l2_loss(params, X, y, lam) -> jax.Array:
    """TODO: bce_loss + lam / 2 * ||w||^2 (no penalty on the bias)."""
    raise NotImplementedError

In [ ]:
lam = 1e-3
log_p, hist_lr = train(bce_l2_loss, init_params(57), Xtr_s, ytr_s.astype(float), lam, lr=0.05, steps=8000)
log_sk = LogisticRegression(C=1 / (lam * len(ytr_s)), tol=1e-10, max_iter=10_000).fit(Xtr_s, ytr_s)
compare("logistic weights", log_p["w"], log_sk.coef_[0])
compare("logistic bias", log_p["b"], log_sk.intercept_[0])
proba = np.asarray(sigmoid(Xte_s @ log_p["w"] + log_p["b"]))
compare("test probabilities", proba, log_sk.predict_proba(Xte_s)[:, 1])
print("ours   ", {k: round(v, 4) for k, v in classification_metrics(yte_s, (proba >= 0.5).astype(int)).items()})
print("sklearn", {k: round(v, 4) for k, v in classification_metrics(yte_s, log_sk.predict(Xte_s)).items()})

### E2. Thresholds and the cost of a mistake

The default threshold 0.5 treats both mistakes alike. Hiding a real e-mail (a false positive) is worse than letting
a spam through. Plot the metrics against the threshold, then pick the **smallest** threshold whose precision on the
*training* set is at least 0.99, and report the confusion matrix and the metrics on the test set.

In [ ]:
plot_thresholds(yte_s, proba, classification_metrics)

proba_tr = np.asarray(sigmoid(Xtr_s @ log_p["w"] + log_p["b"]))
grid = np.linspace(0.05, 0.99, 95)
ok = [t for t in grid if classification_metrics(ytr_s, (proba_tr >= t).astype(int))["precision"] >= 0.99]
t_star = ok[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, t in zip(axes, [0.5, t_star]):
    plot_confusion(*confusion_counts(yte_s, (proba >= t).astype(int)), ax=ax, title=f"threshold {t:.2f}")
plt.show()
for t in (0.5, t_star):
    print(f"threshold {t:.2f}:", {k: round(v, 3) for k, v in classification_metrics(yte_s, (proba >= t).astype(int)).items()})

In [ ]:
# Logistic regression is a linear classifier: XOR needs the polynomial features, as for the perceptron.
Px = polynomial_features(Xx, 2)
p_lx, _ = train(bce_l2_loss, init_params(5), Px, yx.astype(float), 1e-3, lr=0.1, steps=2000)
q_lin, _ = train(bce_l2_loss, init_params(2), Xx, yx.astype(float), 1e-3, lr=0.1, steps=2000)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
plot_boundary(lambda Z: predict_class(q_lin, Z), Xx, yx, "logistic, raw features", ax=axes[0])
plot_boundary(lambda Z: predict_class(p_lx, polynomial_features(Z, 2)), Xx, yx, "logistic, degree 2", ax=axes[1])
plt.show()
print("accuracy: raw features", np.mean(predict_class(q_lin, Xx) == yx), "| degree-2 features", np.mean(predict_class(p_lx, Px) == yx))

## Part F — Challenge: One Protocol, Three Models

Compare **logistic regression on the log features, logistic regression on the raw frequencies and the perceptron**
on spam with the same protocol (guide, Part F): stratified 5-fold cross-validation on the training set, every
preprocessing step fitted **inside** each fold, the mean and the standard deviation of the **MCC** for each model, and a
final test-set report for the best one. Then choose its decision threshold on a validation fold, not on the test set.

In [ ]:
# TODO (home): implement the protocol above and report a table: model, mean MCC, std MCC.